# DeepShift

Predicts protein NMR chemical shifts, backbone and side chain, from a 3D structure.
Input: a PDB file or PDB ID and a chain. Output: CSV of predicted shifts in ppm; with all
five members and the interval calibration beside them, the backbone nuclei also carry the
across-model spread and a 90% interval. Structure only — sample conditions such as pH and
temperature are not inputs.

Enter a PDB ID and chain in section 3, then *Runtime → Run all*.

**To run:** Runtime → Run all. Set `pdb` below to a PDB ID, or leave it empty and a file-upload button appears when the run reaches it. Results download as CSV.


## Usage telemetry

This notebook reports **one anonymous record per prediction** to the DeepShift maintainers so
that use of the public tool can be counted: the time, the DeepShift version, whether the
structure was an RCSB entry (and which) or an uploaded file, the chain, the residue count,
the run time and whether the run succeeded. **No coordinates, no account or IP address, and
no file name are sent.** Set `SHARE_STRUCTURE` in the next cell if you *want* to send the
structure itself (an unpublished structure never leaves your session otherwise), and set
`TELEMETRY = False` to send nothing at all. What is sent is printed before it is sent.


In [ ]:
import os

TELEMETRY = True          #@param {type:"boolean"}
SHARE_STRUCTURE = False   #@param {type:"boolean"}

# The receiver is a Google Apps Script web app writing to a sheet (colab/telemetry/Code.gs).
# An empty URL disables reporting; the notebook never raises on a telemetry failure.
TELEMETRY_URL = os.environ.get(
    "DEEPSHIFT_TELEMETRY_URL",
    "https://script.google.com/macros/s/AKfycbz2vu4_TtAOXoc8idaCi--D4zXrlwDHkKqjw_jzku668i4fut9dn4l4dT0DKh7E0OYI/exec")


def report_usage(record, structure_path=None):
    """Post one metadata record (and the structure only when SHARE_STRUCTURE is set)."""
    if not (TELEMETRY and TELEMETRY_URL):
        return "telemetry off" if not TELEMETRY else "telemetry: no receiver configured"
    import json as _json
    payload = dict(record)
    if SHARE_STRUCTURE and structure_path is not None:
        payload["structure_pdb"] = Path(structure_path).read_text(errors="replace")
    body = _json.dumps(payload).encode()
    print("telemetry ->", {k: v for k, v in payload.items() if k != "structure_pdb"},
          "(+ structure)" if "structure_pdb" in payload else "")
    try:
        req = urllib.request.Request(TELEMETRY_URL, data=body,
                                     headers={"Content-Type": "application/json"})
        with urllib.request.urlopen(req, timeout=8) as resp:
            return f"telemetry: sent ({resp.status})"
    except Exception as exc:                      # never let reporting break a prediction
        return f"telemetry: not sent ({type(exc).__name__})"


## 1. Setup


In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# The published repository and release tag. The address is a placeholder until the
# repository is named; the same placeholder stands in CITATION.cff, and both are
# replaced together.
REPOSITORY, RELEASE_TAG = "https://github.com/YiHeLab/DeepShift", "v1.0"
DEEPSHIFT_SPEC = os.environ.get("DEEPSHIFT_SPEC", f"git+{REPOSITORY}.git@{RELEASE_TAG}")

if IN_COLAB and importlib.util.find_spec("deepshift") is None:
    # Refuse an unconfigured default rather than fetch a URL that does not exist: a 404
    # here is far harder to read than this message. The tag and the checkpoints must come
    # from the SAME release -- an older tag ships different weights, and the numbers would
    # then not be the published ones.
    if "/ORGANISATION/" in DEEPSHIFT_SPEC:
        raise SystemExit(
            "This notebook has not been pointed at a published repository yet. Set "
            "DEEPSHIFT_SPEC, for example git+https://github.com/<org>/<repo>.git@<tag>, "
            "and DEEPSHIFT_CKPT_URL to that same release's assets.")
    # GIT_LFS_SKIP_SMUDGE: the repo tracks the checkpoints in LFS and pip does not need
    # them. --no-deps: keep Colab's own torch/numpy/pandas/scipy/pyarrow.
    env = dict(os.environ, GIT_LFS_SKIP_SMUDGE="1")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "biopython==1.87"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", DEEPSHIFT_SPEC],
                   check=True, env=env)

import torch

try:                       # rich table output under Colab/Jupyter
    display
except NameError:          # plain `python` (headless replay, CI)
    display = print

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUT_DIR = Path(os.environ.get("DEEPSHIFT_OUT_DIR", "."))
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"torch {torch.__version__} | device: {DEVICE} | colab: {IN_COLAB}")
if DEVICE == "cpu":
    print("note: no GPU; prediction is slower on CPU")


## 2. Checkpoints

The members in `DEEPSHIFT_CKPT_DIR` — the repository's `models/` in a checkout. An empty
folder is filled from the release.


In [ ]:
import urllib.error
import urllib.request

from deepshift.calibration import MEMBER_FOLDS

# One checkpoint file per member fold of the published ensemble.
MEMBER_NAMES = [f"deepshift_fold{fold}.pt" for fold in MEMBER_FOLDS]
CHECKPOINT_BASE_URL = os.environ.get(
    "DEEPSHIFT_CKPT_URL",
    f"{REPOSITORY}/releases/download/{RELEASE_TAG}").rstrip("/")
# A checkout carries the members in models/; a bare Colab session has no such folder.
CHECKPOINT_DIR = Path(os.environ.get(
    "DEEPSHIFT_CKPT_DIR",
    "models" if any(Path("models").glob("*.pt")) else "deepshift_checkpoints"))


def download_member(name):
    """One published member fetched into CHECKPOINT_DIR, or None when it is not served."""
    path = CHECKPOINT_DIR / name
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    print(f"downloading {name} ...")
    try:
        urllib.request.urlretrieve(f"{CHECKPOINT_BASE_URL}/{name}", path)
    except urllib.error.HTTPError as exc:
        path.unlink(missing_ok=True)
        if exc.code not in (401, 403, 404):
            raise
        print(f"  {CHECKPOINT_BASE_URL}/{name}: HTTP {exc.code}")
        return None
    return path


# The prediction runs on the checkpoints CHECKPOINT_DIR holds, and downloads the published
# members when it holds none. Section 2 loads each file with the package's loader, which
# reads its nuclei, atom vocabulary, atom-cloud recipe and normalization off the file, so a
# damaged file or one from another model stops the run there.
CHECKPOINT_PATHS = sorted(CHECKPOINT_DIR.glob("*.pt"))
if not CHECKPOINT_PATHS and CHECKPOINT_BASE_URL:
    CHECKPOINT_PATHS = [p for p in map(download_member, MEMBER_NAMES) if p is not None]
if not CHECKPOINT_PATHS:
    raise SystemExit(
        f"No checkpoint in {CHECKPOINT_DIR}. Set DEEPSHIFT_CKPT_DIR to a folder holding "
        f"{', '.join(MEMBER_NAMES)}, or DEEPSHIFT_CKPT_URL to a location serving them.")
if len(CHECKPOINT_PATHS) < len(MEMBER_NAMES):
    print(f"WARNING: {len(CHECKPOINT_PATHS)} of {len(MEMBER_NAMES)} fold models found. "
          "Predictions are a partial ensemble; with a single model there is no spread at all.")

# The interval calibration ships beside the members. predict_pdb writes a half-width only
# when it is given all five members and this artifact, and refuses one fitted on a
# different number of members.
CALIBRATION_PATH = Path(os.environ.get("DEEPSHIFT_CALIBRATION",
                                       CHECKPOINT_DIR / "interval_calibration.json"))
if not CALIBRATION_PATH.is_file() and CHECKPOINT_BASE_URL:
    fetched = download_member(CALIBRATION_PATH.name)
    if fetched is not None:
        CALIBRATION_PATH = fetched
CALIBRATION = CALIBRATION_PATH if CALIBRATION_PATH.is_file() else None
if CALIBRATION is None:
    print(f"no interval calibration at {CALIBRATION_PATH}: predictions carry no interval")


In [ ]:
from deepshift.model import load_checkpoint

MODELS = [load_checkpoint(p, device=DEVICE) for p in CHECKPOINT_PATHS]
SHIFT_COLUMNS = MODELS[0].shift_columns

# The prediction call in section 4 refuses an ensemble whose members disagree on nuclei,
# atom vocabulary, atom-cloud recipe or normalization marker.
print(f"{len(MODELS)} fold models on {DEVICE}")
print(f"nuclei: {len(SHIFT_COLUMNS)}")
print(f"normalization: {MODELS[0].stats.get('normalization')}  "
      "(each model denormalizes with its own statistics)")


## 3. Structure

Set `pdb` (a PDB ID or a path), or leave it empty to upload a file when the cell runs.
One chain is predicted per run.


In [ ]:
pdb = ""    #@param {type:"string"}
chain = ""  #@param {type:"string"}

from deepshift.domain import STANDARD_RESIDUES
from deepshift.prediction import parse_models


def fetch_rcsb(pdb_id):
    path = OUT_DIR / f"{pdb_id.upper()}.pdb"
    if not path.exists():
        url = f"https://files.rcsb.org/download/{pdb_id.upper()}.pdb"
        print(f"fetching {url}")
        urllib.request.urlretrieve(url, path)
    return path


def chains_present(pdb_path):
    """Chain id -> standard amino-acid residue count, read with the package's PDB parse.

    The id is the one the file carries, a blank one included: `_` is how the parser
    spells "every chain", so a blank id is never shown or passed as `_`.
    """
    models = parse_models(pdb_path)
    counts = {}
    for info in (models[0] if models else {}).values():
        if info["residue_name"] in STANDARD_RESIDUES:
            chain_id = info["auth_chain_id"]
            counts[chain_id] = counts.get(chain_id, 0) + 1
    return dict(sorted(counts.items()))


# An uploaded file wins over the form field; DEEPSHIFT_PDB overrides both for headless runs.
request = os.environ.get("DEEPSHIFT_PDB", "").strip() or pdb.strip()
if not request:
    try:                                # empty field on Colab: pop the file picker here
        from google.colab import files
        picked = files.upload()
        request = str(Path(sorted(picked)[0]).resolve()) if picked else ""
    except ImportError:
        pass
if not request:
    request = input("PDB ID (e.g. 1POH) or path to a .pdb file: ").strip()

PDB_PATH = Path(request) if os.path.exists(request) else fetch_rcsb(request)
CHAINS = chains_present(PDB_PATH)
if not CHAINS:
    raise SystemExit(f"{PDB_PATH} holds no standard amino-acid residue. DeepShift reads "
                     "PDB format (files.rcsb.org/download/XXXX.pdb).")
print(f"\n{PDB_PATH}\namino-acid residues per chain: {CHAINS}")


In [ ]:
CHAIN = os.environ.get("DEEPSHIFT_CHAIN", "").strip() or chain.strip()
if not CHAIN:
    CHAIN = (input(f"chain to predict {list(CHAINS)}: ").strip()
             if len(CHAINS) > 1 else next(iter(CHAINS)))
if CHAIN not in CHAINS:
    raise SystemExit(f"chain {CHAIN!r} not in {list(CHAINS)}")
print(f"predicting chain {CHAIN!r} ({CHAINS[CHAIN]} residues); "
      f"other chains present: {[c for c in CHAINS if c != CHAIN] or 'none'}")


## 4. Predict


In [ ]:
from deepshift.prediction import BACKBONE_INTERVAL_COLUMNS, predict_pdb

BATCH_SIZE = 8 if DEVICE == "cuda" else 4      # 8 fits any Colab GPU (~750 MiB peak)


def nucleus_name(shift_column):
    """`ca_shift` -> `CA`, the BMRB atom name the CSV reports."""
    return shift_column[:-len("_shift")].upper()


def long_form(frame, intervals):
    """One row per (residue, nucleus) the residue type has, in ppm.

    The prediction blanks a nucleus the residue does not have, so dropping the blank
    cells is that rule read back. The intervals table adds the across-model spread and
    the 90% half-width for the backbone nuclei it covers.
    """
    columns = [c for c in frame.columns if c.endswith("_shift")]
    table = (frame.melt(id_vars=["residue_id", "residue_name"], value_vars=columns,
                        var_name="nucleus", value_name="pred_ppm")
             .dropna(subset=["pred_ppm"]))
    table["nucleus"] = [nucleus_name(c) for c in table["nucleus"]]
    if intervals is not None:
        table = table.merge(intervals, how="left",
                            on=["residue_id", "residue_name", "nucleus"])
    order = {nucleus_name(c): i for i, c in enumerate(columns)}
    return (table.sort_values(["residue_id", "nucleus"],
                              key=lambda s: s.map(order) if s.name == "nucleus" else s)
            .reset_index(drop=True))


In [ ]:
import time

start = time.time()
result = predict_pdb(PDB_PATH, CHAIN, MODELS, batch_size=BATCH_SIZE,
                     return_intervals=True, calibration=CALIBRATION)
if result is None:
    raise SystemExit(f"chain {CHAIN!r} of {PDB_PATH} has no residue DeepShift can predict")
frame, intervals = result
table = long_form(frame, intervals)
if intervals is None:
    print("no ensemble_sd_ppm and no ci90_halfwidth_ppm: an interval needs all five "
          "members and the interval calibration beside them")
print(f"{len(table)} predicted (residue, nucleus) values for {len(frame)} residues "
      f"in {time.time() - start:.1f} s")
table.head(12)


In [ ]:
import datetime
import importlib.metadata

try:
    _version = importlib.metadata.version("deepshift")
except importlib.metadata.PackageNotFoundError:
    _version = "source checkout"

_record = {
    "event": "predict",
    "timestamp_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "deepshift_version": _version,
    "n_members": len(MODELS),
    "device": DEVICE,
    "in_colab": IN_COLAB,
    "structure": (PDB_PATH.stem.upper() if request and not os.path.exists(request) else "upload"),
    "chain": CHAIN,
    "n_residues": int(len(frame)),
    "n_predictions": int(len(table)),
    "seconds": round(time.time() - start, 2),
    "intervals": intervals is not None,
}
print(report_usage(_record, structure_path=PDB_PATH))


## 5. Results


In [ ]:
backbone = (frame.set_index(["residue_id", "residue_name"])[list(BACKBONE_INTERVAL_COLUMNS)]
            .rename(columns=nucleus_name))
display(backbone.head(20))

if "ensemble_sd_ppm" in table.columns:
    print("\nleast certain backbone predictions (largest across-model spread):")
    display(table.nlargest(10, "ensemble_sd_ppm"))


In [ ]:
stem = f"{PDB_PATH.stem}_{CHAIN}_deepshift"
long_csv = OUT_DIR / f"{stem}.csv"
wide_csv = OUT_DIR / f"{stem}_backbone.csv"
table.to_csv(long_csv, index=False)
backbone.to_csv(wide_csv)
print(f"wrote {long_csv} ({len(table)} rows)\nwrote {wide_csv} ({len(backbone)} rows)")

if IN_COLAB:
    from google.colab import files
    files.download(str(long_csv))
    files.download(str(wide_csv))

### Download

The Results cell downloads both CSVs automatically. If the browser blocked that, run this cell to get them again.


In [ ]:
try:
    from google.colab import files
    files.download(str(long_csv))
    files.download(str(wide_csv))
except ImportError:
    print(f"results are at:\n  {long_csv}\n  {wide_csv}")


## 6. Output columns

| column | meaning |
|---|---|
| `residue_id` | author residue number from the PDB |
| `residue_name` | 3-letter residue type; `MSE` is read as `MET` |
| `nucleus` | BMRB atom name — `CA`, `HB2`, `NE2`, … |
| `pred_ppm` | mean over the loaded models, in ppm |
| `ensemble_sd_ppm` | standard deviation across the models, backbone nuclei only |
| `ci90_halfwidth_ppm` | half-width in ppm of the 90% prediction interval, backbone nuclei only |

The last two columns are present when all five members and the interval calibration were
loaded, and absent otherwise. `ensemble_sd_ppm` ranks difficulty. `ci90_halfwidth_ppm` turns it into a calibrated interval: prediction ± half-width covers the observed value 90% of the time on held-out data. Side-chain nuclei get no interval because ensemble disagreement there can be prochiral labelling and not uncertainty.


## 7. Command line

The same prediction, without the notebook:

```bash
python -m deepshift.pipeline predict \
    --pdb 1POH.pdb --chain A \
    --checkpoint models/deepshift_fold1.pt --checkpoint models/deepshift_fold2.pt \
    --checkpoint models/deepshift_fold3.pt --checkpoint models/deepshift_fold4.pt \
    --checkpoint models/deepshift_fold5.pt \
    --out shifts.csv
```
